# POLARIS v3 -- Multi-Agent AI Governance Engine

### *An OpenEnv RL Environment for Training LLMs on Multi-Agent Negotiation, Theory-of-Mind, and Long-Horizon Planning*

**Built solo by [Abhishek A S](https://github.com/abhishekascodes) | Age 17**

| Stat | Value |
|------|-------|
| **GRPO Improvement** | +126.3% reward in 13 min |
| **Ministers** | 5 LLM-powered personas |
| **State Metrics** | 21 interconnected variables |
| **Tasks** | 6 difficulty levels |
| **Llama 70B** | Collapses 0.96 to 0.22 |

In [ ]:
# Install dependencies
!pip install -q openenv-core torch transformers trl accelerate datasets matplotlib openai fastapi uvicorn pydantic pyyaml

In [ ]:
# Clone the repository
!git clone https://github.com/abhishekascodes/POLARIS-V3.git
import os
os.chdir('POLARIS-V3')
print('POLARIS v3 cloned successfully')

## 1. Environment Overview
POLARIS simulates a nation with 21 interconnected metrics. The agent governs by choosing policy actions while navigating a council of 5 ministers who negotiate, form coalitions, and veto decisions.

In [ ]:
import sys
sys.path.insert(0, '.')
from server.policy_environment import PolicyEnvironment
from server.config import TASK_CONFIGS, VALID_ACTIONS

print('POLARIS v3 -- Environment Summary')
print('=' * 60)
print(f'  Tasks:   {len(TASK_CONFIGS)}')
print(f'  Actions: {len(VALID_ACTIONS)}')
print()
for tid, cfg in TASK_CONFIGS.items():
    neg = '[NEG]' if cfg.get('enable_negotiation') else '     '
    print(f'  {neg} {tid:30s} max_steps={cfg["max_steps"]}')
    print(f'       {cfg["description"]}')
    print()

## 2. Run an Episode -- Full Negotiation Arena
Watch the agent interact with 5 ministers, form coalitions, and face vetoes.

In [ ]:
import random

env = PolicyEnvironment()
obs = env.reset(seed=42, task_id='negotiation_arena')
actions = sorted(list(VALID_ACTIONS))

print('NEGOTIATION ARENA -- Live Episode')
print('=' * 70)

total_reward = 0
vetoes_seen = 0

for step in range(30):
    if obs.done:
        print(f'\nCOLLAPSED at step {step}')
        break
    
    meta = obs.metadata
    sat = meta.get('public_satisfaction', 50)
    poll = meta.get('pollution_index', 100)
    gdp = meta.get('gdp_index', 100)
    
    # Heuristic policy
    if sat < 25: action = 'increase_welfare'
    elif poll > 200: action = 'enforce_emission_limits'
    elif gdp < 50: action = 'stimulate_economy'
    else: action = actions[step % len(actions)]
    
    action_data = {
        'action': action, 'reasoning': f'Step {step}',
        'coalition_target': ['Chancellor Voss'],
        'veto_prediction': ['Dr. Vasquez'] if poll > 150 else [],
        'stance': 'cooperative'
    }
    
    obs = env.step(action_data)
    total_reward += obs.reward
    
    outcome = meta.get('negotiation_outcome', {})
    if outcome.get('vetoed'): vetoes_seen += 1
    
    if step % 5 == 0:
        narrative = meta.get('negotiation_narrative', '')
        print(f'\n  Step {step:2d} | {action:25s} | R={obs.reward:+.2f} | GDP={gdp:.0f} Poll={poll:.0f} Sat={sat:.0f}')
        if narrative:
            print(f'         Council: {narrative[:120]}...')

collapsed = obs.metadata.get('collapsed', False)
print(f'\n{"=" * 70}')
print(f'  Total Reward: {total_reward:.2f}')
print(f'  Status: {"COLLAPSED" if collapsed else "SURVIVED"}')
print(f'  Vetoes encountered: {vetoes_seen}')

## 3. GRPO Training Results
Qwen 2.5 3B trained with curriculum-weighted ToM reward on RTX 5080.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('POLARIS v3 -- Training Results (Qwen 3B, RTX 5080)', fontsize=14, fontweight='bold')
fig.patch.set_facecolor('#fafafa')

# 1. Before vs After
ax = axes[0]
bars = ax.bar(['Before', 'After\nGRPO'], [13.4, 30.2], color=['#a1a1aa', '#4f46e5'], width=0.5, edgecolor='white', linewidth=2)
for b, v in zip(bars, [13.4, 30.2]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.5, f'{v}', ha='center', fontweight='bold', fontsize=14)
ax.set_ylabel('Avg Episode Reward')
ax.set_title('Reward: +126.3%', fontweight='bold', color='#059669')
ax.grid(axis='y', alpha=0.3); ax.set_facecolor('#fafafa')

# 2. Curriculum
ax = axes[1]
labels = ['Easy\n3/3', 'Medium\n2/3', 'Hard\n0/3', 'Extreme\n0/3']
vals = [40.8, 38.3, 24.9, 22.7]
colors = ['#059669', '#d97706', '#e11d48', '#7c3aed']
bars = ax.bar(labels, vals, color=colors, width=0.5, edgecolor='white', linewidth=2)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.5, f'{v}', ha='center', fontweight='bold', fontsize=11)
ax.set_ylabel('Avg Reward')
ax.set_title('Curriculum Escalation', fontweight='bold')
ax.grid(axis='y', alpha=0.3); ax.set_facecolor('#fafafa')

# 3. Llama 70B Collapse
ax = axes[2]
tasks = ['Easy', 'Balanced', 'Hard\nGov', 'Multi\nAgent', 'Negot\nArena']
scores = [0.96, 0.15, 0.17, 0.20, 0.22]
bar_colors = ['#059669'] + ['#e11d48']*4
bars = ax.bar(tasks, scores, color=bar_colors, width=0.5, edgecolor='white', linewidth=2)
for b, v in zip(bars, scores):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02, f'{v}', ha='center', fontweight='bold', fontsize=10)
ax.set_ylabel('Score')
ax.set_title('Llama 70B: 77% Collapse', fontweight='bold', color='#e11d48')
ax.axhline(y=0.3, color='gray', linestyle='--', alpha=0.3)
ax.grid(axis='y', alpha=0.3); ax.set_facecolor('#fafafa')

plt.tight_layout()
plt.savefig('polaris_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSummary:')
print('  +126.3% reward improvement (13.4 -> 30.2) in 13 minutes')
print('  Curriculum: Easy 40.8 (3/3) -> Extreme 22.7 (0/3)')
print('  Llama 70B collapses from 0.96 to 0.22 under negotiation pressure')
print('  Theory-of-Mind accuracy: 0% for frontier LLMs')

## 4. Reward System Deep Dive

In [ ]:
print('POLARIS v3 -- 6-Component Composite Reward')
print('=' * 55)
components = [
    ('Base Governance',    '~40%', 'Multi-metric improvement (GDP, satisfaction, etc)'),
    ('Pareto Optimality',  '~15%', 'Balanced improvement across ALL dimensions'),
    ('Theory-of-Mind',     '~15%', 'Correct veto predictions -> social cognition'),
    ('Coalition Bonus',    '~10%', 'Successfully forming minister coalitions'),
    ('Briefing Compliance','~10%', 'Acting on timed intelligence before deadlines'),
    ('Oscillation Penalty','~10%', 'Penalizes flip-flopping between actions'),
]
for name, weight, desc in components:
    print(f'  {name:>22s} [{weight}] -- {desc}')

print('\nCurriculum ToM Training Pipeline:')
print('  Phase 1 (0-40%):  Heavy ToM weight -> Teach veto prediction')
print('  Phase 2 (40-70%): Balanced -> Multi-objective learning')
print('  Phase 3 (70-100%): Governance-dominant -> Stabilize policy')

## 5. Try It Yourself -- Run Multiple Tasks

In [ ]:
print('Running all 6 tasks with heuristic policy...\n')

for task_id in TASK_CONFIGS:
    env = PolicyEnvironment()
    obs = env.reset(seed=42, task_id=task_id)
    ep_reward = 0
    steps = 0
    max_s = TASK_CONFIGS[task_id]['max_steps']
    
    for step in range(min(max_s, 50)):
        if obs.done: break
        steps = step
        meta = obs.metadata
        sat = meta.get('public_satisfaction', 50)
        poll = meta.get('pollution_index', 100)
        gdp = meta.get('gdp_index', 100)
        
        if sat < 25: action = 'increase_welfare'
        elif poll > 200: action = 'enforce_emission_limits'
        elif gdp < 50: action = 'stimulate_economy'
        else: action = actions[step % len(actions)]
        
        obs = env.step({'action': action, 'reasoning': 'heuristic',
                       'coalition_target': [], 'veto_prediction': [], 'stance': 'cooperative'})
        ep_reward += obs.reward
    
    status = 'SURVIVED' if not obs.metadata.get('collapsed') else 'COLLAPSED'
    neg = '[NEG]' if TASK_CONFIGS[task_id].get('enable_negotiation') else '     '
    print(f'  {neg} {task_id:30s} reward={ep_reward:7.2f} steps={steps:3d} {status}')

---

## Links

| Resource | Link |
|----------|------|
| **GitHub** | [POLARIS-V3](https://github.com/abhishekascodes/POLARIS-V3) |
| **HF Space** | [asabhishek/polaris-v3](https://huggingface.co/spaces/asabhishek/polaris-v3) |
| **API Docs** | [/docs endpoint](https://asabhishek-polaris-v3.hf.space/docs) |

---

*Built for the Meta PyTorch OpenEnv Hackathon x Scaler 2026*